In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:90% ! important;}
div.cell.code_cell.rendered{width:100%}
div.input_prompt{padding:0px}
div.CodeMirror {font-family:Consolas ; font-size:12pt;}
div.text_cell_render.rendered_html {font-size:12pt;}
div.output {font-size:12pt; font-weight:bold}
div.input {font-family:Consolas ; font-size:12pt;}
div.prompt {min-width:70px;}
div#toc-wrapper {padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:12px;}
</style>
"""))

<b><font size="6" color="red">Ollama_LLM으로 EXAONE 4.0 1.2B 사용(LnagChain)</font></b>
# 테스트

In [3]:
# %conda install -c conda-forge openpyxl

In [23]:
from langchain_ollama import ChatOllama
llm = ChatOllama(model = 'sam860/exaone-4.0:1.2b')

In [24]:
import pandas as pd
data = pd.read_excel('data/test_labeling.xlsx')

In [5]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 557 entries, 0 to 556
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   날짜      557 non-null    datetime64[ns]
 1   제목      557 non-null    object        
 2   본문      555 non-null    object        
 3   링크      557 non-null    object        
 4   산업      557 non-null    object        
dtypes: datetime64[ns](1), object(4)
memory usage: 21.9+ KB


In [6]:
data.loc[0,'본문']

'미국 11월 소비자물가지수(CPI)가 전년 대비 2.7% 상승했다고 18일(현지시간) 미국 노동통계국이 밝혔다. 이는 전문가 전망치(3.1%)와 전월치(2.9%)를 모두 밑도는 수치이자 2021년 초 이후 가장 낮은 수준이다. 변동성이 큰 식품과 에너지를 제외한 근원 CPI는 전년 대비 2.6% 상승했다.'

In [7]:
result = llm.invoke(data.loc[0,'본문']+'의 내용을 요약해 줘')

In [8]:
test_data = result.content

In [9]:
test_talk = llm.invoke(test_data+'의 내용은 긍정적인지 부정적인지만 파악해 줘')

In [10]:
test_talk.content

'주어진 정보에 따르면, 11월 미국 소비자물가지수(CPI)가 전년 대비 **2.7% 상승**한 것은 전체 인플레이션률에서는 높은 수치였으나, 전문가 전망치(3.1%)보다 낮았고 전월치(2.9%)에도 미치지 못했습니다. 이로 인해 변동성이 큰 식품·에너지 제외 근원 CPI는 더 낮은 증가율(2.6%)을 기록하며 완화된 인플레이션 압력을 보였습니다.  \n\n### 종합적 평가:  \n1. **부정적 요소**:  \n   - 전반적인 CPI 상승이 전문가 예상보다 낮아 경제적 우려가 일부 존재합니다. 특히 에너지·식품 등 핵심 항목의 큰 하락으로 전체 물가 안정에는 긍정적 측면도 있었지만, 중장기적 기준으로는 추가적인 통화정책 필요성을 시사할 수 있습니다.  \n   - 2021년 이후 최저 수준이라는 점에서 소비 및 생산 비용 상승 억제 추세가 지속되고 있음을 반영합니다.  \n\n2. **긍정적 요소**:  \n   - 다른 근원(비식·비에너지)의 점진적인 조정으로 물가 변동성 감소 효과가 나타났습니다. 이는 중앙은행 정책 부담 완화로 이어질 수 있습니다.  \n\n결론적으로, 단기적으로는 완화된 인플레이션이지만, 구조적 문제나 글로벌 공급망 재편 등 배경 요인을 고려할 때 지속적인 모니터링이 필요합니다.'

In [23]:
test_file=pd.read_csv('data/테스트파일.csv')

test_file.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 72 entries, 0 to 71
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   본문      72 non-null     object
 1   산업      72 non-null     object
dtypes: object(2)
memory usage: 1.2+ KB


In [12]:
results = []
for text in test_file[:10]['본문']:
    result = llm.invoke(text + " 내용을 요약하고 긍정적인지 부정적인지만 파악해 줘")
    results.append(result.content)

In [20]:
test_results=[]
for idx in range(len(results)):
    result = llm.invoke(results[idx] + '내용중에 긍정적인지 부정적인지 두 방향으로 나누지 말고 하나의 시선으로만 파악해줘 예를 들어 이 기사가 부정적인것 같다면 부정적이고 그 이유를 요약하는 식으로')
    test_results.append(result.content)

In [21]:
for idx in range(len(test_results)):
    print(test_results[idx])
    print("-"*50)

**감정 분석 요약: 부정적 측면 강조**  

해당 사고는 대규모 인명 피해와 구조적 결함으로 인해 명확한 책임 소재 논란을 낳았습니다. 기업의 공식 사과에도 불구하고, 원인 규명 과정에서 표면과 150m 거리에서 진행된 연결 통로 문제 등 근본적 위험 관리 소홀이 확인되면서 사회적 신뢰 회복에 어려움을 겪고 있습니다. 특히 작업 중 sudden structural failure(철근 붕괴)가 발생한 점과 인명 피해 규모를 고려할 때, 단순한 과실 이상의 시스템적 취약성 노출로 평가됩니다. 정부와 기업의 재발 방지 대책 마련 여부도 향후 유사 사고 예방력에 대한 의문을 남긴다.
--------------------------------------------------
이 사고는 인명 피해와 더불어 공사 지연, 교통 혼란 등 중대한 사회적·경제적 파장을 예상케 하는 사건이다. 특히 대형 크레인 사고로 인한 매몰 인부의 희생은 산업 현장에 대한 불신을 deepen시키고, 긴급 복구 비용이 추가될 우려가 커 부정적 결과가 계속될 가능성이 매우 높다.
--------------------------------------------------
### 사고 평가 (부정적 관점 중심)  

본 사고는 건설 현장에서 발생한 중대한 인명사고로, 철근 구조물 붕�라는 구조적 결함이 원인으로 지목되면서 근본적인 예방 시스템의 취약성과 관리 소홀 문제가 드러납니다. 신속한 대응은 필요했으나, 감리단의 초기 확인 미흡과 보수적인 수리 지시 등이 피해자 회복 과정에서 추가적 고통을 야기할 수 있었습니다. 특히 외국인 작업자 지원 미비와 경제적 손실 우려는 사회적 책임 측면에서도 우려되며, 단순한 응급 조치를 넘어 안전 문화 개선 방안 마련 없이는 재발 가능성이 높습니다. 현장 조사 완료 후에도 책임 소재 규명과 공정 개선이 지연될 경우 향후 유사 사고 리스크가 지속될 수 있어 근본적 보완이 시급합니다.
---------------------------------------------

In [24]:
results = []
for text in test_file['본문']:
    result = llm.invoke(text + '내용중에 긍정적인지 부정적인지 두 방향으로 나누지 말고 하나의 시선으로만 파악해줘 예를 들어 이 기사가 부정적인것 같다면 부정적이고 그 이유를 요약하는 식으로')
    results.append(result.content)

In [16]:
for idx in range(len(results)):
    print(results[idx])
    print("-"*50)

In [28]:
result_sentiment=[]
for idx in range(len(results)):
    result = llm.invoke(results[idx]+'내용을 읽고 긍정/중립/부정 중 하나의 단어를 선택해 줘. 예를 들어 긍정적인 내용이면 "긍정"만 출력하면 돼')
    result_sentiment.append(result.content)

In [48]:
sentiment_replace=[]
for idx in range(len(result_sentiment)):
    sentiment_replace.append(result_sentiment[idx].replace('"',"").replace(" ",""))
print(sentiment_replace)

['부정', '부정', '부정', '긍정', '긍정', '긍정', '부정', '긍정', '부정', '긍정', '긍정', '긍정', '긍정', '긍정', '긍정', '긍정', '부정', '부정', '긍정', '긍정', '부정', '긍정', '부정', '긍정', '긍정', '긍정', '긍정', '강화', '긍정', '긍정', '긍정', '부정', '부정', '긍정', '부정', '부정', '긍정', '긍정', '긍정', '부정', '부정', '긍정', '긍정', '긍정', '긍정', '긍정', '긍정', '긍정', '긍정', '긍정', '긍정', '긍정', '부정', '긍정', '증명', '긍정', '긍정', '긍정', '긍정', '긍정', 'positive', '긍정', '긍정', '긍정', '긍정', 'positive', '긍정', '긍정', '긍정', 'positive', '긍정', '긍정']


In [51]:
set(sentiment_replace)

{'positive', '강화', '긍정', '부정', '증명'}

In [4]:
import csv

In [6]:
with open("data/감정분석전처리전.csv", "w", newline="\n", encoding="utf-8") as f: 
    writer = csv.writer(f) 
    writer.writerow(sentiment_replace)

In [11]:
with open('data/감정분석전처리전.csv','r', encoding='utf-8') as f:
    reader = csv.reader(f) 
    test_data = next(reader) 
print(test_data)

['부정', '부정', '부정', '긍정', '긍정', '긍정', '부정', '긍정', '부정', '긍정', '긍정', '긍정', '긍정', '긍정', '긍정', '긍정', '부정', '부정', '긍정', '긍정', '부정', '긍정', '부정', '긍정', '긍정', '긍정', '긍정', '강화', '긍정', '긍정', '긍정', '부정', '부정', '긍정', '부정', '부정', '긍정', '긍정', '긍정', '부정', '부정', '긍정', '긍정', '긍정', '긍정', '긍정', '긍정', '긍정', '긍정', '긍정', '긍정', '긍정', '부정', '긍정', '증명', '긍정', '긍정', '긍정', '긍정', '긍정', 'positive', '긍정', '긍정', '긍정', '긍정', 'positive', '긍정', '긍정', '긍정', 'positive', '긍정', '긍정']


In [14]:
sentiment_result=[]
for idx in range(len(test_data)):
    if test_data[idx] not in('강화','증명','positive','negative'):
        sentiment_result.append(test_data[idx])
    elif test_data[idx] in('강화', 'positive'):
        sentiment_result.append('긍정')
    elif test_data[idx] == 'negative':
        sentiment_result.append('부정')
    else:
        sentiment_result.append('중립')

In [15]:
print(sentiment_result)

['부정', '부정', '부정', '긍정', '긍정', '긍정', '부정', '긍정', '부정', '긍정', '긍정', '긍정', '긍정', '긍정', '긍정', '긍정', '부정', '부정', '긍정', '긍정', '부정', '긍정', '부정', '긍정', '긍정', '긍정', '긍정', '긍정', '긍정', '긍정', '긍정', '부정', '부정', '긍정', '부정', '부정', '긍정', '긍정', '긍정', '부정', '부정', '긍정', '긍정', '긍정', '긍정', '긍정', '긍정', '긍정', '긍정', '긍정', '긍정', '긍정', '부정', '긍정', '중립', '긍정', '긍정', '긍정', '긍정', '긍정', '긍정', '긍정', '긍정', '긍정', '긍정', '긍정', '긍정', '긍정', '긍정', '긍정', '긍정', '긍정']


In [58]:
with open("data/감정분석전처리테스트.csv", "w", newline="\n", encoding="utf-8") as f: 
    writer = csv.writer(f) 
    writer.writerow(sentiment_result)

In [28]:
import pickle

# 피클 파일 불러오기
with open("data/648_20251201-20251221.pkl", "rb") as f:   # 'rb' = read binary
    loaded_data = pickle.load(f)

print(loaded_data)
# {'name': 'Alice', 'age': 25, 'hobbies': ['reading', 'cycling']}

                     날짜                                          제목  \
0   2025-12-01 06:36:00             [기자수첩]'주식빚투' 괜찮다?…부동산 '부메랑' 된다면   
1   2025-12-01 06:50:00         석화 구조조정 윤곽 잡히긴 하는데…여수·울산 '지지부진' 이유는   
2   2025-12-01 07:00:00                좁아진 '휠라' 입지…'안타 효과'로 돌파구 찾을까   
3   2025-12-01 07:10:00  [거버넌스워치] 가온그룹, 창업주 작고 뒤 28살 CEO 경영권 안정 안간힘   
4   2025-12-01 07:30:00    [거버넌스워치] ‘캐리어 에어컨’ 오텍그룹, “경영 참여” 주요주주 출현   
..                  ...                                         ...   
663 2025-12-21 13:00:00                      '말차' 유행인데…물 대신 마셔도 되나요   
664 2025-12-21 14:00:00                   'AI 날개' 달고 스마트 철도 시대 앞당긴다   
665 2025-12-21 15:00:00              SDV 전면에 세운 현대차…하드웨어 한계 넘으려는 까닭   
666 2025-12-21 16:00:00                [생활경제25]'K뷰티' 날았다…새 대장은 에이피알   
667 2025-12-21 16:25:00              고려아연·영풍, 美제련소 프로젝트 놓고 연일 공방 지속   

                                                    본문  \
0    올해 코스피는 4000포인트라는 전인미답의 고지를 밟았다. '코스피 5000' 달성...   
1    대산을 시작으로 석유화학업계의 대규모 구조조정이

# 실행코드

In [65]:
from langchain_ollama import ChatOllama
import csv
llm = ChatOllama(model = 'sam860/exaone-4.0:1.2b')
import pandas as pd
data = pd.read_pickle('data/industry_labeled.pickle')

In [66]:
data.reset_index(inplace=True)

In [67]:
data.drop(columns=['index'], inplace=True)

In [32]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9544 entries, 0 to 9543
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   날짜      9544 non-null   datetime64[ns]
 1   제목      9544 non-null   object        
 2   본문      9544 non-null   object        
 3   링크      9544 non-null   object        
 4   산업      9544 non-null   object        
dtypes: datetime64[ns](1), object(4)
memory usage: 372.9+ KB


In [33]:
data.head(10)

,날짜,제목,본문,링크,산업
0,2025-12-01 04:00:00,"이어지는 건설업 규제·처벌 강화…""사후 규제영향평가 도입해야""",최근 국회에서 규제와 처벌 강화를 골자로 하는 건설산업 관련 법안을 대거 입법하는 ...,https://n.news.naver.com/mnews/article/008/000...,건설
1,2025-12-01 04:06:00,유럽 흐르는 제네시스 마그마… 고성능 브랜드로 뜨겁게 경쟁,출범 10년을 맞은 제네시스가 '마그마'를 앞세워 유럽시장 공략에 나선다. 고성능 ...,https://n.news.naver.com/mnews/article/008/000...,자동차
2,2025-12-01 04:09:00,"'닥터나우 방지법' 처리 임박… 벤처업계 ""혁신 족쇄"" 호소",일명 '닥터나우 방지법'(약사법 개정안)이 12월2일 국회 본회의에서 처리될 전망이...,https://n.news.naver.com/mnews/article/008/000...,헬스케어
3,2025-12-01 05:01:00,"[백상논단] 생명공학기술, 철저한 검증과 관리 필요하다",[서울경제] 자기 자손은 유전적으로 완벽해 건강하게 장수하기를 바라는 마음은 모든...,https://n.news.naver.com/mnews/article/011/000...,헬스케어
4,2025-12-01 05:29:00,5년 만에 10배 ‘폭풍성장’ 테슬라...“수입차 점유율 사상 첫 20% 돌파 전망”,[파이낸셜뉴스] 테슬라의 누적 판매 대수가 5년 만에 10배 가까이 늘어나는 등 국...,https://n.news.naver.com/mnews/article/014/000...,자동차
5,2025-12-01 05:46:00,"""친환경은 물론 층간소음도 제로…모듈러 건축, 학교·빌딩·아파트도 가능하죠""",혁신은 위기 속에서 피어난다. 치열한 시장 경쟁과 불확실한 글로벌 경제 환경 속에서...,https://n.news.naver.com/mnews/article/018/000...,건설
6,2025-12-01 05:47:00,"""해외 누비는 건설사, 경제 '국가대표'…그들 돕는 '국가대표' 로펌 될 것""","""해외에서 큰 프로젝트를 수주하는 건설사들은 우리나라 경제를 이끄는 '국가대표'잖아...",https://n.news.naver.com/mnews/article/008/000...,건설
7,2025-12-01 06:01:00,12월 첫째 주 IPO 대전…車보안·AI·조선·바이오 한꺼번에 쏟아진다 [이번주 증...,[서울경제] 12월 첫째 주(1~5일) 기업공개(IPO) 시장에서는 티엠씨(유가증...,https://n.news.naver.com/mnews/article/011/000...,헬스케어
8,2025-12-01 06:01:00,줄도산 위기 아니었나?… 잇달아 회생절차 졸업하는 건설사,"올해만 건설사 9곳 법정관리 신청했지만 대우산업개발, 올해 첫 회생절차 종결 신동아...",https://n.news.naver.com/mnews/article/366/000...,건설
9,2025-12-01 06:02:00,"[사이언스샷] 꿈의 CAR-T 항암제, 자가면역 질환도 잡는다","美FDA, 2017년부터 혈액암 대상 7종 허가 면역과잉이 부른 자가면역 질환도 공...",https://n.news.naver.com/mnews/article/366/000...,헬스케어


In [68]:
# 감정분석을 위해 본문열만 저장
sentimenting_data = data.iloc[:,2]

In [69]:
# 본문열을 리스트로 저장
sentimenting_list = [data for data in sentimenting_data]

In [7]:
# 시간이 오래걸리는 듯 하여 tag_list로 본문 분할 후 감정분석 예정
tag_list = [[] for _ in range(int(len(sentimenting_list)/100))]

for idx in range(int(len(sentimenting_list)/100)):
    for tag in range(100):
        tag_list[idx].append(sentimenting_list[idx*100 + tag])

In [24]:
len(tag_list[:])

95

In [19]:
# 우선 24일에 실행해 볼 코드
# for idx in range(len(tag_list[0])):
result_test=[]
for idx in range(10):
    result = llm.invoke(tag_list[0][idx]+'내용을 읽고 긍정적/중립적/부정적 중 하나의 단어를 선택해 줘. 예를 들어 긍정적인 내용이면 "긍정적"만 출력하면 돼')
    result_test.append(result.content)

In [20]:
result_test

['부정적', '긍정적', '긍정적', '중립적', '긍정적', '"긍정적"', '긍정적', '긍정적', '중립적', '긍정적']

In [16]:
result_test

['부정', '긍정', '부정', '부정', '긍정', '긍정', '긍정', '긍정', '긍정', 'positive']

In [22]:
# tag_list[0][2]

In [35]:
# 저장된 본문들을 하나씩 감정분석(23일 한번에 하려다 시간을 허비함, 시간을 줄이기 위해 리스트를 나눠서 할 필요성이 있어보임)
result_sentiment=[]
for tag_idx in range(len(tag_list[:])):
    for idx in range(100):
        result = llm.invoke(tag_list[tag_idx][idx]+'내용을 읽고 긍정적/중립적/부정적 중 하나의 단어를 선택해 줘. 예를 들어 긍정적인 내용이면 "긍정적"만 출력하면 돼')
        result_sentiment.append(result.content)
    print(f'{tag_idx}번 리스트 작업 감정분석 완료')

In [34]:
result_sentimentiment

##  감정분석된(중간처리 필요) 파일 전처리

In [ ]:
with open("data/피클감정분석전처리테스트.csv", "w", newline="\n", encoding="utf-8") as f: 
    writer = csv.writer(f) 
    writer.writerow(result_sentimentiment)

In [5]:
with open('data/피클감정분석전처리전2.csv','r', encoding='utf-8') as f:
    reader = csv.reader(f) 
    test_data = next(reader) 
len(test_data)

9544

In [13]:
test_list=[]
for idx in range(len(test_data)):
    test_list.append(test_data[idx].replace('"',"").replace(" ","").replace('*','').replace('[','').replace('-',''))
test_set=set(test_list)
test_set

{'5.KAIST가내년부터‘AI단과대학’을신설하고해당정원도학부100명을포함해300명늘리기로했습니다.정부는KAIST를시작으로2027년까지광주과학기술원(GIST),대구경북과학기술원(DGIST),울산과학기술원(UNIST)등다른3대과기원에도AI단과대학을순차적으로설립해‘AI인재양성벨트’를구축하기로했다.',
 'CEO레코그니션',
 'POSCO홀딩스(005490)=자회사포스코이앤씨중대재해발생부정적',
 'negative',
 'neutral',
 'positive',
 '▲전문가들의분석과지적이포함된점을고려할때,이번사태는복잡한기술적·보안적리스크를드러낸부정적사건으로판단됩니다.\n부정적',
 '강화되다',
 '강화된',
 '개발완료,화재감지센서등탑재,공간효율성과이용편의성향상,화재위험선제적차단목표',
 '개발했다',
 '개발했다.',
 '계약체결',
 '규제강화에도불구하고수도권주요지역아파트값상승세지속,실수요자에게실질적대안마련가능',
 '글로벌통상환경의불확실성과각국규제강화등급변하는시장환경',
 '긍정적',
 '기술력및성장잠재력인정',
 "김대중시장'서울시민들이어떤지역에서든쾌적하게살수있도록주택을공급하겠다'.",
 '김명섭◇상무보',
 '대유에이텍(002880)=신규차생산부지109억원에취득내용을읽고긍정적중하나의단어를선택해줘.',
 '디알텍은글로벌벤더블디텍터시장을선도하는기업으로,연간30조원규모의연매출을바탕으로비파괴검사분야에혁신적인디지털방사선솔루션을제공하고있습니다.주요내용을요약하면다음과같습니다:\n\n1.글로벌톱티어업체공급\n디알텍은NDT(NonDestructiveTesting)시장의최상위권파트너사로지정되었으며,벤더블디텍터개발및생산역량을활용해기존아날로그필름기반기술에대한경쟁력있는대안을제시했습니다.\n\n2.기술혁신핵심feature\n곡면촬영시발생하던영상왜곡문제해결:자유로운곡률변형설계로평판형감지기대비선명도와정확도향상.\n저방사량기술:배관용접검사시기존대비10분의1수준의방사선량으로안전성확보가능.\n\n3.ODM생산확대및사업전략\n2024년상반기까지해당업체에맞춤형벤더블디텍터ODM(직접생산manu

In [38]:
# 테스트에선 긍정/중립/부정 외에 다른 단어들도 출력되어 긍정/중립/부정만 나오도록 단어를 수정하기위한 코드(아직 위의 코드의 실행결과를 모르기에 실행결과 확인 후 변경할 것)
sentiment_result=[]
for idx in range(len(test_list)):
    #if test_data[idx] not in('강화','증명','positive','negative'):
    #   sentiment_result.append(test_list[idx])
    if test_list[idx] in('긍정','강화','positive','상승','상향', '증가', '강세'):
        sentiment_result.append('긍정적')
    if test_list[idx] in('부정','negative','사망', '개선', '축소','부정적'):
        sentiment_result.append('부정적')
    elif test_list[idx] in('중립'):
        sentiment_result.append('중립적')
    else:
        sentiment_result.append(test_list[idx])

In [39]:
len(set(sentiment_result))

68

In [37]:
set(sentiment_result)

{'5.KAIST가내년부터‘AI단과대학’을신설하고해당정원도학부100명을포함해300명늘리기로했습니다.정부는KAIST를시작으로2027년까지광주과학기술원(GIST),대구경북과학기술원(DGIST),울산과학기술원(UNIST)등다른3대과기원에도AI단과대학을순차적으로설립해‘AI인재양성벨트’를구축하기로했다.',
 'CEO레코그니션',
 'POSCO홀딩스(005490)=자회사포스코이앤씨중대재해발생부정적',
 'neutral',
 '▲전문가들의분석과지적이포함된점을고려할때,이번사태는복잡한기술적·보안적리스크를드러낸부정적사건으로판단됩니다.\n부정적',
 '강화되다',
 '강화된',
 '개발완료,화재감지센서등탑재,공간효율성과이용편의성향상,화재위험선제적차단목표',
 '개발했다',
 '개발했다.',
 '계약체결',
 '규제강화에도불구하고수도권주요지역아파트값상승세지속,실수요자에게실질적대안마련가능',
 '글로벌통상환경의불확실성과각국규제강화등급변하는시장환경',
 '긍정적',
 '기술력및성장잠재력인정',
 "김대중시장'서울시민들이어떤지역에서든쾌적하게살수있도록주택을공급하겠다'.",
 '김명섭◇상무보',
 '대유에이텍(002880)=신규차생산부지109억원에취득내용을읽고긍정적중하나의단어를선택해줘.',
 '디알텍은글로벌벤더블디텍터시장을선도하는기업으로,연간30조원규모의연매출을바탕으로비파괴검사분야에혁신적인디지털방사선솔루션을제공하고있습니다.주요내용을요약하면다음과같습니다:\n\n1.글로벌톱티어업체공급\n디알텍은NDT(NonDestructiveTesting)시장의최상위권파트너사로지정되었으며,벤더블디텍터개발및생산역량을활용해기존아날로그필름기반기술에대한경쟁력있는대안을제시했습니다.\n\n2.기술혁신핵심feature\n곡면촬영시발생하던영상왜곡문제해결:자유로운곡률변형설계로평판형감지기대비선명도와정확도향상.\n저방사량기술:배관용접검사시기존대비10분의1수준의방사선량으로안전성확보가능.\n\n3.ODM생산확대및사업전략\n2024년상반기까지해당업체에맞춤형벤더블디텍터ODM(직접생산manufacturing)제품완성후공급시작계획을발표했습

In [56]:
p_keywords = ['긍정','강화','positive','상승','상향', '증가', '강세','긍정적','계약체결','개발','인정','수상','계약체결']
ng_keywords = ['부정','negative','사망', '개선', '축소','부정적']
nt_keywords = ['중립', '중립적', 'neutral']
new_value = ['긍정적','중립적',"부정적"]

my_list = [new_value[0] if any(k in str(word) for k in p_keywords) 
           else new_value[1] if any(k in str(word) for k in nt_keywords)
           else new_value[2] if any(k in str(word) for k in ng_keywords)
           else word
           for word in test_list]

set(my_list)

{'5.KAIST가내년부터‘AI단과대학’을신설하고해당정원도학부100명을포함해300명늘리기로했습니다.정부는KAIST를시작으로2027년까지광주과학기술원(GIST),대구경북과학기술원(DGIST),울산과학기술원(UNIST)등다른3대과기원에도AI단과대학을순차적으로설립해‘AI인재양성벨트’를구축하기로했다.',
 'CEO레코그니션',
 '긍정적',
 "김대중시장'서울시민들이어떤지역에서든쾌적하게살수있도록주택을공급하겠다'.",
 '김명섭◇상무보',
 '면역',
 '부정적',
 '선행',
 '신뢰성확보',
 '열기',
 '예방',
 '예상',
 '윤',
 '음성',
 '음성평가결과의주요지표인등급부여기준',
 '전문가권고안(PONENTE)',
 '절차진행',
 '정박',
 '중도금무이자혜택제공단지희소성이부각된다는점에서',
 '중립적',
 '지원한다',
 '최고등급',
 '파트너사들과긴밀한협력을바탕으로',
 '합리적',
 '협약체결',
 '획득'}

In [57]:
len(set(my_list))

26

In [63]:
# 재감정분석이 필요한 데이터
index = [ my_list.index('5.KAIST가내년부터‘AI단과대학’을신설하고해당정원도학부100명을포함해300명늘리기로했습니다.정부는KAIST를시작으로2027년까지광주과학기술원(GIST),대구경북과학기술원(DGIST),울산과학기술원(UNIST)등다른3대과기원에도AI단과대학을순차적으로설립해‘AI인재양성벨트’를구축하기로했다.'),
          my_list.index('CEO레코그니션'),
          my_list.index("김대중시장'서울시민들이어떤지역에서든쾌적하게살수있도록주택을공급하겠다'."),
          my_list.index('김명섭◇상무보'),
          my_list.index('면역'),
          my_list.index('윤'),
          my_list.index('음성'),
          my_list.index('음성평가결과의주요지표인등급부여기준'),
          my_list.index('전문가권고안(PONENTE)'),
          my_list.index('절차진행'),
          my_list.index('정박'),
          my_list.index('중도금무이자혜택제공단지희소성이부각된다는점에서'),
          my_list.index('지원한다'),
          my_list.index('최고등급'),
          my_list.index('파트너사들과긴밀한협력을바탕으로'),
          my_list.index('합리적'),
          my_list.index('협약체결'),
          my_list.index('획득')
         ]
index.sort()

In [87]:
len(index)

18

In [84]:
re_sentimente_list=[]
for i in range(len(index)):
    re_sentimente_list.append(sentimenting_list[index[i]])
re_sentimente_list

In [74]:
re_result_test=[]
for idx in range(len(index)):
    result = llm.invoke(re_sentimente_list[idx]+'내용을 읽고 긍정적/중립적/부정적 중 하나의 단어를 선택해 줘. 다른 의견을 내지 말고 저 3단어 중 하나로만 하는거야')
    re_result_test.append(result.content)

In [75]:
re_result_test

['긍정적',
 'content_type: "analysis"  \nsummary_word: **실행력**',
 '평가 결과',
 '김병민 서울시 정무부시장: 공공시설 통합 및 쾌적한 주거환경 구현',
 '효율적',
 '안전성',
 '긍정적',
 '1. 긍정적',
 '효과적',
 '긍정적',
 '경제',
 ' 합리적',
 '승진',
 '중도금 무이자',
 '친환경',
 '긍정적',
 '긍정적',
 '긍정적']

In [81]:
re_sentimente_list[14]

'한국타이어앤테크놀로지의 대전공장이 친환경 국제인증인 ‘ISCC PLUS(International Sustainability & Carbon Certification Plus)’를 획득했다고 19일 밝혔다.  ISCC PLUS는 바이오 기반 및 재활용 원료의 지속가능성과 공급망 투명성을 검증하는 국제 인증 제도다. 금산공장과 헝가리 라칼마스 공장에 이은 세 번째 인증으로, 주요 글로벌 생산거점 전반에 지속가능 원료 기반 생산체계를 확산하고 있다는 점에서 의미가 있다.  한국타이어는 대전공장에서 석유화학 합성고무를 바이오-서큘러 폴리머로 대체해 원료 조달부터 생산 전 과정에서 환경 발자국을 줄이며 엄격한 인증 기준을 충족했다. 이를 통해 신차용, 교체용 타이어는 물론 모터스포츠용 제품까지 아우르는 친환경 생산체계를 한층 강화했다.  특히 ISCC PLUS 인증 원료를 포함한 지속가능 원료를 최대 31% 적용한 고성능 레이싱 타이어는 국제자동차연맹(FIA) 주관 월드 랠리 챔피언십(WRC) 2025 시즌부터 공식 타이어로 독점 공급되며 친환경 기술력과 성능 경쟁력을 동시에 입증하고 있다.  한국타이어는 ISCC PLUS 인증 원료를 매스밸런스 방식 기준 최대 77%까지 높인 신제품 ‘아이온 GT’를 유럽 교체용 시장에 출시하기도 했다. 아이온 GT는 전기차 전용 타이어 최초로 ‘EU 타이어 라벨 등급’ 회전저항, 젖은 노면 접지력, 소음 3개 부문에서 ‘트리플 A’ 등급을 획득했다.  한국타이어 관계자는 “순환경제 전략을 바탕으로 석유 자원 의존도를 낮추고 천연자원 고갈을 방지하고, 탄소 배출량을 지속적으로 감축해 타이어 산업의 지속가능성 제고에 일조할 계획”이라고 전했다.'

In [82]:
# 마지막까지 감정분석이 되지 않은 데이터들은 본문 확인 후 주관적으로 판단함
re_result_test[1]='긍정적'
re_result_test[2]='중립적'
re_result_test[3]='긍정적'
re_result_test[4]='긍정적'
re_result_test[5]='긍정적'
re_result_test[7]='긍정적'
re_result_test[8]='부정적'
re_result_test[10]='긍정적'
re_result_test[11]='긍정적'
re_result_test[12]='중립적'
re_result_test[13]='긍정적'
re_result_test[14]='긍정적'

In [86]:
len(re_result_test)

18

In [88]:
for i in range(len(index)):
    my_list[index[i]]=re_result_test[i]
set(my_list)

{'긍정적', '부정적', '선행', '신뢰성확보', '열기', '예방', '예상', '중립적', '협약체결'}

In [89]:
index2 = [my_list.index('선행'),
          my_list.index('신뢰성확보'),
          my_list.index('열기'),
          my_list.index('예방'),
          my_list.index('예상'),
          my_list.index('협약체결'),
         ]

In [90]:
index2

[52, 6897, 2690, 3151, 8477, 6930]

In [102]:
re_sentimente_list2=[]
for i in range(len(index2)):
    re_sentimente_list2.append(sentimenting_list[index2[i]])
# re_sentimente_list2

In [94]:
re_result_test2=[]
for idx in range(len(index2)):
    result = llm.invoke(re_sentimente_list2[idx]+'내용을 읽고 긍정적/중립적/부정적 중 하나의 단어를 선택해 줘. 다른 의견을 내지 말고 저 3단어 중 하나로만 하는거야')
    re_result_test2.append(result.content)

In [95]:
re_result_test2

[' positive', '신뢰성', '중립적', '중립적', '긍정적', '협약']

In [100]:
re_sentimente_list2[-1]

'[스타트업에 대한 보다 다양한 기업정보는 유니콘팩토리 빅데이터 플랫폼 \'데이터랩\'에서 볼 수 있습니다.]  AI(인공지능) 기반 뇌 건강 솔루션 기업 에스에이치엠디(SHMD)가 미국의 한인 생명과학자 네트워크 \'K-BioX\'와 업무협약을 체결했다고 15일 밝혔다.  K-BioX는 미국을 중심으로 활동하는 생명과학자·의료진·바이오산업 종사자들이 참여하는 비영리 전문 네트워크다. 국제 공동 연구, 기술 상용화 지원, 글로벌 바이오 생태계 구축을 선도하고 있다.  양측은 이번 협약을 통해 △데이터 기반 글로벌 연구 및 임상 협력 △전문가 멘토링 및 기술 자문 △해외 시장 진출 전략 공유 △혁신 스타트업 생태계 연계 등 다각적인 협력 프로그램을 추진한다.  SHMD는 헬스케어 전문 법조인이자 개발자인 송민영 대표가 지난해 6월 창업한 스타트업으로, 웨어러블 초음파 기반 뇌혈류 측정 기기 \'세레밴드\'(Cereband)와 AI 기반 뇌건강 분석 앱 \'브레인체크\'(BrainCheck)를 개발했다.  브레인체크는 스마트폰만으로도 치매·뇌졸중을 비롯한 뇌건강 패턴을 해석할 수 있다. 지난 3월 출시 후 10여개 의료기관에서 활용되며, 두통·어지럼증·인지 저하 등 신경계 증상을 겪는 환자의 컨디션 평가에 도움을 주고 있다.  세레밴드는 국내 주요 대학병원과 글로벌 의료기관을 중심으로 PoC(기술검증)가 진행 중이다. SHMD는 K-BioX와의 파트너십을 바탕으로 핵심 기술을 글로벌 임상 환경에서 검증하고, 해외 의료·연구기관과의 협업 네트워크를 확장한다는 목표다.  SHMD는 지난 9월 미국 팔로알토에 현지 사무소를 설립했으며, 내년부터 북미·유럽·동남아·남미 등 주요 시장에서 세레밴드 PoC 확대 및 브레인체크 글로벌 출시을 순차적으로 진행할 계획이다.  송민영 SHMD 대표는 "세계 각국 의료·연구 기관들과의 협력을 확장하는 중요한 전환점이 될 것"이라며 "AI 기반 뇌혈류 헬스케어 분야에서 국제 전문가들과의 공동 연구, 임상 협력, 북미 시장을 포함한 글로

In [101]:
re_result_test2[0]='긍정적'
re_result_test2[1]='긍정적'
re_result_test2[-1]='긍정적'

In [103]:
for i in range(len(index2)):
    my_list[index2[i]]=re_result_test2[i]
set(my_list)

{'긍정적', '부정적', '중립적'}

In [104]:
len(my_list)

9544

In [105]:
data['감정평가'] = my_list

In [118]:
# 날짜열에 깨진 값이 있는지 날짜형태로 불러오는 게 오류가 나는 경우가 생겨 데이터 형식을 날짜형으로 재변환
df["날짜"] = pd.to_datetime(df["날짜"], errors="coerce")

data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9544 entries, 0 to 9543
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   날짜      9544 non-null   datetime64[ns]
 1   제목      9544 non-null   object        
 2   본문      9544 non-null   object        
 3   링크      9544 non-null   object        
 4   산업      9544 non-null   object        
 5   감정평가    9544 non-null   object        
dtypes: datetime64[ns](1), object(5)
memory usage: 447.5+ KB


In [145]:
data.to_csv('data/감정분석결과_엑사온.csv')

In [109]:
df=pd.read_csv('data/감정분석결과.csv')

In [143]:
# 건설 하루(12/1)치 기사
target_date = pd.to_datetime("2025-12-08").date()
target_category = "건설"

dta_testing_list = df[(df["날짜"].dt.date == target_date) & (df['산업']==target_category)]
len(dta_testing_list)

91

In [144]:
dtp_testing_list = df[(df["날짜"].dt.date == target_date) & (df['산업']==target_category)&(df['감정평가']=='긍정적')]
len(dtp_testing_list)

80

In [150]:
chk_list = df[df['감정평가']=='긍정적']
len(chk_list)

8670